# 🔭 Vision Transformer (ViT) – Wizner
**Projekt: Vision Transformer vs CNN | GRUPA 3**

## Zadanie na następną lekcję
Implementacja uproszczonego Vision Transformer dla CIFAR-10:
- [ ] **Patch Embedding** – podziel obraz na patche i zamień na tokeny
- [ ] **Positional Encoding** – dodaj informację o pozycji patchy
- [ ] **Multi-Head Self-Attention** – mechanizm uwagi
- [ ] **Transformer Encoder** – NX warstw enkodera (Attention + FFN)
- [ ] **Klasyfikacja** – [CLS] token → głowa klasyfikacyjna
- [ ] Stworzyć funkcję `run_vit(epochs, NX, num_heads, patch_size, data)` → wyniki

## Architektura ViT (uproszczona)
```
Obraz (3×32×32)
  → Patch Embedding  (patch_size×patch_size → d_model)
  → + Positional Encoding
  → + [CLS] token
  → Transformer Encoder × NX
       └─ Multi-Head Self-Attention (num_heads)
       └─ Feed-Forward Network
  → CLS token → Linear → num_classes
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Patch Embedding

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Dzieli obraz na patche i rzutuje każdy patch na wektor d_model.
    Dla CIFAR-10 (32×32) z patch_size=4: mamy (32/4)^2 = 64 patche.
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3, d_model=128):
        super().__init__()
        assert img_size % patch_size == 0, "img_size musi być podzielne przez patch_size"

        self.num_patches = (img_size // patch_size) ** 2
        self.patch_dim   = in_channels * patch_size * patch_size

        # Projekcja: patch_dim → d_model
        # TODO: użyj nn.Conv2d z kernel_size=patch_size i stride=patch_size
        #       to jest bardziej efektywne niż reshape + Linear
        self.projection = nn.Conv2d(
            in_channels, d_model,
            kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x):
        # x: (batch, C, H, W)
        x = self.projection(x)          # (batch, d_model, H/P, W/P)
        x = x.flatten(2)                # (batch, d_model, num_patches)
        x = x.transpose(1, 2)          # (batch, num_patches, d_model)
        return x

## 2. Transformer Encoder

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Jedna warstwa enkodera: Multi-Head Attention + FFN + LayerNorm."""
    def __init__(self, d_model=128, num_heads=4, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, num_heads,
                                            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * mlp_ratio, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Pre-Norm (bardziej stabilne od Post-Norm)
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out                 # residual
        x = x + self.ffn(self.norm2(x))  # residual
        return x

## 3. Pełny model ViT

In [ ]:
class SimpleViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 d_model=128, num_heads=4, num_layers=4,
                 num_classes=10, dropout=0.1):
        """
        Args:
            img_size   : rozmiar obrazu (zakładamy kwadrat)
            patch_size : rozmiar patcha (NX w zadaniu)
            d_model    : wymiar embeddingu
            num_heads  : liczba głów atencji
            num_layers : liczba warstw enkodera (NX w zadaniu)
            num_classes: liczba klas
        """
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)
        num_patches      = self.patch_embed.num_patches

        # Learnable [CLS] token i positional encoding
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.dropout = nn.Dropout(dropout)

        # Stos warstw enkodera
        self.encoder = nn.Sequential(*[
            TransformerEncoderBlock(d_model, num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.norm       = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B = x.shape[0]

        # Patch embedding
        x = self.patch_embed(x)                        # (B, num_patches, d_model)

        # Dodaj [CLS] token
        cls = self.cls_token.expand(B, -1, -1)         # (B, 1, d_model)
        x   = torch.cat([cls, x], dim=1)               # (B, num_patches+1, d_model)

        # Dodaj positional encoding
        x = x + self.pos_embed
        x = self.dropout(x)

        # Enkoder
        x = self.encoder(x)                            # (B, num_patches+1, d_model)
        x = self.norm(x)

        # Klasyfikacja przez [CLS] token
        cls_out = x[:, 0]                              # (B, d_model)
        return self.classifier(cls_out)                # (B, num_classes)

## 4. Funkcja parametryczna run_vit()

In [ ]:
def run_vit(epochs, num_layers, num_heads, patch_size,
            train_loader, test_loader, num_classes=10,
            d_model=128, lr=3e-4):
    """
    Trenuje ViT z podanymi hiperparametrami.

    Args:
        epochs      : liczba epok
        num_layers  : liczba warstw enkodera (NX)
        num_heads   : liczba głów atencji
        patch_size  : rozmiar patcha (musi dzielić 32)

    Returns:
        dict: test_acc, train_acc, train_time_s, num_params, history
    """
    model = SimpleViT(
        patch_size=patch_size,
        num_heads=num_heads,
        num_layers=num_layers,
        num_classes=num_classes,
        d_model=d_model
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Liczba parametrów: {num_params:,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = []
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        correct_train, total_train = 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

        scheduler.step()
        train_acc = 100 * correct_train / total_train

        model.eval()
        correct_test, total_test = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                correct_test += (predicted == labels).sum().item()
                total_test += labels.size(0)

        test_acc = 100 * correct_test / total_test
        avg_loss = running_loss / len(train_loader)
        print(f"Epoka [{epoch}/{epochs}] | Loss: {avg_loss:.4f} | "
              f"Train: {train_acc:.2f}% | Test: {test_acc:.2f}%")
        history.append({'epoch': epoch, 'train_acc': train_acc,
                        'test_acc': test_acc, 'loss': avg_loss})

    train_time = time.time() - start_time
    print(f"\nTrening zakończony w czasie: {train_time:.2f}s")

    return {
        'test_acc':     test_acc,
        'train_acc':    train_acc,
        'train_time_s': train_time,
        'num_params':   num_params,
        'history':      history,
    }

## 5. Szybki test – sprawdź kształty tensorów

In [ ]:
# Test kształtów (bez treningu, tylko forward pass)
model_test = SimpleViT(patch_size=4, num_heads=4, num_layers=2).to(device)
dummy = torch.randn(2, 3, 32, 32).to(device)   # batch=2, CIFAR-10
out   = model_test(dummy)
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")   # oczekiwane: (2, 10)
print(f"Parametry: {sum(p.numel() for p in model_test.parameters()):,}")

## TODO – do zrobienia przed lekcją
1. Uruchom test kształtów i sprawdź czy output to (2, 10)
2. Załaduj dane (CIFAR-10) i uruchom `run_vit(epochs=2, num_layers=4, num_heads=4, patch_size=4, ...)`
3. Sprawdź wpływ `patch_size`: przetestuj patch_size=2, 4, 8 (ile patchy wychodzi?)
4. Porównaj liczbę parametrów ViT vs CNN (z Welyczko_CNN_v2.ipynb)